In [5]:
import re
import scipy.stats as stats

# 1. Lê o arquivo de texto com a tabela LaTeX
with open("latex_results_table.txt", "r", encoding="utf-8") as f:
    latex_table = f.read()

qwen_data = {"File": [], "Class": [], "Method": [], "Block": []}
litem_data = {"File": [], "Class": [], "Method": [], "Block": []}

def extrai_f1(texto_coluna):
    """Limpa o texto da coluna do LaTeX e extrai o primeiro número flutuante encontrado."""
    # Remove \textbf{...} ou \cellcolor{...}
    limpo = re.sub(r'\\[a-zA-Z!0-9]+(\{([^}]+)\})*', r'\2', texto_coluna)
    match = re.search(r'\d+\.\d+', limpo)
    return float(match.group()) if match else None

# 2. Processa linha por linha dividindo pelas colunas (&)
for line in latex_table.split('\n'):
    # Ignora explicitamente a linha de médias para não distorcer o teste estatístico
    if "Average" in line or "cellcolor" in line:
        continue
        
    if "Qwen2.5-7B-Instruct" in line or "LiteM" in line:
        colunas = line.split('&')
        if len(colunas) >= 18:  # Uma linha válida tem o projeto, modelo e as 16 métricas
            # No formato da tabela, os F1-scores estão fixos nestas colunas (índices baseados em 0):
            # File F1 = col 4, Class F1 = col 8, Method F1 = col 12, Block F1 = col 16
            f1_file = extrai_f1(colunas[4])
            f1_class = extrai_f1(colunas[8])
            f1_method = extrai_f1(colunas[12])
            f1_block = extrai_f1(colunas[16])
            
            # Garante que capturou todos antes de salvar
            if all(v is not None for v in [f1_file, f1_class, f1_method, f1_block]):
                target_dict = qwen_data if "Qwen2.5-7B-Instruct" in line else litem_data
                target_dict["File"].append(f1_file)
                target_dict["Class"].append(f1_class)
                target_dict["Method"].append(f1_method)
                target_dict["Block"].append(f1_block)

# 3. Executa os testes para todas as granularidades extraídas
granularities = ["File", "Class", "Method", "Block"]

for g in granularities:
    qw = qwen_data[g]
    li = litem_data[g]
    
    print(f"\n=== GRANULARIDADE: {g.upper()} (Amostras capturadas: {len(qw)}) ===")
    
    if len(qw) != len(li) or len(qw) == 0:
        print(f"  Erro: Tamanho incompatível de amostras! Qwen: {len(qw)}, LiteM: {len(li)}")
        continue
        
    # Teste de Normalidade
    shap_q = stats.shapiro(qw).pvalue
    shap_l = stats.shapiro(li).pvalue
    print(f"  Shapiro-Wilk Qwen (p-valor): {shap_q:.4f}")
    print(f"  Shapiro-Wilk LiteM (p-valor): {shap_l:.4f}")
    
    # Seleção automática do teste
    if shap_q >= 0.05 and shap_l >= 0.05:
        stat, p_val = stats.ttest_rel(qw, li)
        test_name = "Paired T-Test (Paramétrico)"
    else:
        stat, p_val = stats.wilcoxon(qw, li)
        test_name = "Wilcoxon Signed-Rank (Não-paramétrico)"
        
    print(f"  Teste Escolhido: {test_name}")
    print(f"  p-valor Final: {p_val}")
    print(f"  Significativo (alpha=0.05)? {'SIM' if p_val < 0.05 else 'NÃO'}")


=== GRANULARIDADE: FILE (Amostras capturadas: 18) ===
  Shapiro-Wilk Qwen (p-valor): 0.3399
  Shapiro-Wilk LiteM (p-valor): 0.6898
  Teste Escolhido: Paired T-Test (Paramétrico)
  p-valor Final: 7.707457081748922e-05
  Significativo (alpha=0.05)? SIM

=== GRANULARIDADE: CLASS (Amostras capturadas: 18) ===
  Shapiro-Wilk Qwen (p-valor): 0.3326
  Shapiro-Wilk LiteM (p-valor): 0.0477
  Teste Escolhido: Wilcoxon Signed-Rank (Não-paramétrico)
  p-valor Final: 0.00029247830381213256
  Significativo (alpha=0.05)? SIM

=== GRANULARIDADE: METHOD (Amostras capturadas: 18) ===
  Shapiro-Wilk Qwen (p-valor): 0.2317
  Shapiro-Wilk LiteM (p-valor): 0.0227
  Teste Escolhido: Wilcoxon Signed-Rank (Não-paramétrico)
  p-valor Final: 7.62939453125e-06
  Significativo (alpha=0.05)? SIM

=== GRANULARIDADE: BLOCK (Amostras capturadas: 18) ===
  Shapiro-Wilk Qwen (p-valor): 0.9311
  Shapiro-Wilk LiteM (p-valor): 0.0047
  Teste Escolhido: Wilcoxon Signed-Rank (Não-paramétrico)
  p-valor Final: 0.000712626702